# Model 3 - 3C: ResNet50 + LoRA

Transfer learning: trọng số ResNet50 vẫn đóng băng, nhưng thêm adapter LoRA (tự viết) vào các lớp Conv2d của layer3 và layer4. Chỉ train adapter + head MLP.

Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation, test và lưu model.

In [1]:
# 1. Import và cấu hình
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, precision_recall_fscore_support

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / 'data').exists() and PROJECT_DIR != PROJECT_DIR.parent:
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data' / 'VN_trash_classification_preprocessing'
CHECKPOINT_DIR = PROJECT_DIR / 'src' / 'model3' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Trọng số ImageNet được tải về (hoặc đã có sẵn) trong src/model3/pretrained
torch.hub.set_dir(PROJECT_DIR / 'src' / 'model3' / 'pretrained' / 'hub')

BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 7
HEAD_LR = 1e-3
LORA_LR = 1e-3
WEIGHT_DECAY = 1e-4
LORA_RANK = 8
LORA_ALPHA = 16.0
SEED = 42
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# 2. Xử lý ảnh
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

In [3]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = ImageFolder(DATA_DIR / 'val', transform=test_transform)
test_dataset = ImageFolder(DATA_DIR / 'test', transform=test_transform)

# drop_last=True: tránh batch chỉ có 1 ảnh làm BatchNorm1d của head báo lỗi
# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

class_names = train_dataset.classes
print('Classes:', class_names)
print('Train:', len(train_dataset), '| Validation:', len(val_dataset), '| Test:', len(test_dataset))

Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
Train: 9542 | Validation: 1685 | Test: 864


In [4]:
# 4. Tạo model: backbone ImageNet + LoRA + head MLP
class LoRAConv2d(nn.Module):
    # y = conv_goc(x) + (alpha / rank) * up(down(x)), conv gốc đóng băng, chỉ down/up được train
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        for parameter in self.base.parameters():
            parameter.requires_grad = False

        self.scaling = alpha / rank
        self.lora_down = nn.Conv2d(
            base.in_channels,
            rank,
            kernel_size=base.kernel_size,
            stride=base.stride,
            padding=base.padding,
            dilation=base.dilation,
            bias=False,
        )
        self.lora_up = nn.Conv2d(rank, base.out_channels, kernel_size=1, bias=False)

        nn.init.kaiming_uniform_(self.lora_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_up.weight)  # lora_up = 0 nên lúc đầu adapter chưa đổi gì

    def forward(self, images):
        return self.base(images) + self.scaling * self.lora_up(self.lora_down(images))


class TransferModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()  # giữ BatchNorm của backbone cố định, adapter vẫn học bình thường
        return self

    def forward(self, images):
        return self.head(self.backbone(images))


# ResNet50 pretrained ImageNet, bỏ lớp fc cuối (2048 -> 1000) để lấy vector 2048 chiều
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
backbone.fc = nn.Identity()
feature_dim = 2048

# Đóng băng toàn bộ backbone
for parameter in backbone.parameters():
    parameter.requires_grad = False

# Thay mọi Conv2d trong layer3 và layer4 bằng LoRAConv2d (adapter mới mặc định có requires_grad=True)
for name, module in list(backbone.named_modules()):
    if name.startswith(('layer3', 'layer4')):
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Conv2d):
                setattr(module, child_name, LoRAConv2d(child, LORA_RANK, LORA_ALPHA))

head = nn.Sequential(
    nn.Linear(feature_dim, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(512, len(class_names)),
)

model = TransferModel(backbone, head).to(device)
loss_function = nn.CrossEntropyLoss()

lora_parameters = [parameter for parameter in model.backbone.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(
    [
        {'params': model.head.parameters(), 'lr': HEAD_LR},
        {'params': lora_parameters, 'lr': LORA_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

Trainable params: 1,570,825 / 25,078,857


In [5]:
# 5. Train và Validation
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_function(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_val_accuracy = 0.0
best_weights = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step()

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

# Vòng lặp kết thúc khi model đang giữ trọng số của epoch cuối, nạp lại trọng số tốt nhất.
model.load_state_dict(best_weights)
print(f'Best epoch: {best_epoch} | Best Val Acc: {best_val_accuracy:.2%}')

Epoch 1/30 | Train Loss: 0.4138 | Train Acc: 86.43% | Val Loss: 0.2431 | Val Acc: 92.28%


Epoch 2/30 | Train Loss: 0.1816 | Train Acc: 94.23% | Val Loss: 0.2314 | Val Acc: 92.94%


Epoch 3/30 | Train Loss: 0.1490 | Train Acc: 95.12% | Val Loss: 0.2308 | Val Acc: 93.12%


Epoch 4/30 | Train Loss: 0.1082 | Train Acc: 96.43% | Val Loss: 0.2360 | Val Acc: 92.70%


Epoch 5/30 | Train Loss: 0.1006 | Train Acc: 96.68% | Val Loss: 0.3780 | Val Acc: 89.55%


...

Epoch 26/30 | Train Loss: 0.0008 | Train Acc: 99.97% | Val Loss: 0.2366 | Val Acc: 95.31%


Epoch 27/30 | Train Loss: 0.0008 | Train Acc: 99.97% | Val Loss: 0.2378 | Val Acc: 95.43%


Epoch 28/30 | Train Loss: 0.0009 | Train Acc: 99.98% | Val Loss: 0.2314 | Val Acc: 95.43%


Epoch 29/30 | Train Loss: 0.0006 | Train Acc: 100.00% | Val Loss: 0.2333 | Val Acc: 95.31%


Epoch 30/30 | Train Loss: 0.0004 | Train Acc: 100.00% | Val Loss: 0.2380 | Val Acc: 95.25%
Best epoch: 27 | Best Val Acc: 95.43%


In [6]:
# 6. Test
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        true_labels += labels.tolist()
        predicted_labels += outputs.argmax(dim=1).cpu().tolist()

accuracy = sum(t == p for t, p in zip(true_labels, predicted_labels)) / len(true_labels)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='macro', zero_division=0
)
_, _, weighted_f1, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='weighted', zero_division=0
)

print()
print('Test Result')
print('Test Accuracy:', f'{accuracy:.2%}')
print('Macro Precision:', f'{macro_precision:.4f}')
print('Macro Recall:', f'{macro_recall:.4f}')
print('Macro F1:', f'{macro_f1:.4f}')
print('Weighted F1:', f'{weighted_f1:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 96.53%
Macro Precision: 0.9635
Macro Recall: 0.9658
Macro F1: 0.9643
Weighted F1: 0.9653

              precision    recall  f1-score   support

         Alu     0.9896    0.9596    0.9744        99
      Carton     0.9903    0.9444    0.9668       108
    Foam_box     0.9425    0.9425    0.9425        87
    Milk_box     0.9901    0.9901    0.9901       101
       Other     0.9245    0.9800    0.9515        50
         PET     0.9897    0.9412    0.9648       102
       Paper     0.9369    0.9811    0.9585       106
   Paper_cup     0.9722    1.0000    0.9859       105
 Plastic_cup     0.9352    0.9528    0.9439       106

    accuracy                         0.9653       864
   macro avg     0.9635    0.9658    0.9643       864
weighted avg     0.9660    0.9653    0.9653       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR / '3c_lora_resnet50.pt')